# Perbaikan SENA CPU v0.5

Eksperimen baru dari bobot baseline SENA v1 yang tersedia, bukan reproduksi bobot kandidat Colab yang belum ditemukan. Menggunakan development split baru dari 730 label manusia yang sudah diketahui dan 64 pasangan latihan berlabel asisten. Validation lama 620 dipensiunkan sebagai untouched validation. Pilot 50 tidak dipakai untuk training atau pemilihan checkpoint.

BERT dibekukan kecuali dua layer akhir, pooler dan classifier. Konfigurasi, kriteria seleksi dan penghentian dini dibekukan sebelum run. Hasil development dan diagnostik yang telah dilihat tidak dinyatakan sebagai test penerimaan independen. Target mutu tidak diturunkan.


In [1]:
from pathlib import Path
import os,sys,json,subprocess,hashlib
ROOT = Path(os.environ['SLR09_PACKAGE_ROOT'])
ASSETS = Path(os.environ['SLR09_ASSET_DIR'])
OUTPUT = Path(os.environ['SLR09_TRAIN_OUTPUT'])
print('Python:',sys.version)
print('Bertolak dari baseline SENA v1; tidak memakai pilot50 untuk fit/seleksi.')
print('Source SHA:',hashlib.sha256((ROOT/'code/quality_fix/train_sena_cpu.py').read_bytes()).hexdigest())


Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Bertolak dari baseline SENA v1; tidak memakai pilot50 untuk fit/seleksi.
Source SHA: 75d418e0929631b4126413c7d326168c141bf7f7142f375ba98d65cc39e0bae5


In [2]:
command=[sys.executable,str(ROOT/'code/quality_fix/train_sena_cpu.py'),
 '--root',str(ROOT),'--archive',str(ASSETS/'SILAR_SENA_v1.zip'),'--output',str(OUTPUT)]
completed=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in completed.stdout:
 print(line,end='',flush=True)
completed.wait()
if completed.returncode:raise RuntimeError('Training tidak selesai; periksa run_state dan log, jangan klaim lulus.')


epoch 1 step 1/81 loss 1.1779
epoch 1 step 10/81 loss 2.6458
epoch 1 step 20/81 loss 0.8087
epoch 1 step 30/81 loss 0.6531
epoch 1 step 40/81 loss 0.5476
epoch 1 step 50/81 loss 0.5163
epoch 1 step 60/81 loss 0.5749
epoch 1 step 70/81 loss 0.7876
epoch 1 step 80/81 loss 0.1351
development 1 {"accuracy": 0.8287671232876712, "macro_f1": 0.8240230833480411, "per_class": {"FLOOD": {"support": 32, "precision": 0.8285714285714286, "recall": 0.90625, "f1": 0.8656716417910447}, "EARTHQUAKE": {"support": 24, "precision": 0.8148148148148148, "recall": 0.9166666666666666, "f1": 0.8627450980392157}, "FIRE": {"support": 35, "precision": 0.9166666666666666, "recall": 0.9428571428571428, "f1": 0.9295774647887324}, "ACCIDENT": {"support": 24, "precision": 0.9166666666666666, "recall": 0.9166666666666666, "f1": 0.9166666666666666}, "NON_EVENT": {"support": 31, "precision": 0.625, "recall": 0.4838709677419355, "f1": 0.5454545454545454}}, "predeclared_selection_metric": 0.5376344086021505, "targets_met":

In [3]:
state=json.loads((OUTPUT/'run_state.json').read_text())
metrics=json.loads((OUTPUT/'best_development_metrics.json').read_text())
print(json.dumps(state,indent=2))
print(json.dumps(metrics,indent=2))
assert state['training_executed'] and state['status']=='SUCCEEDED'
print('Checkpoint dipilih pada development validation. Penerimaan mutu akhir belum diberikan.')


{
  "status": "SUCCEEDED",
  "training_executed": true,
  "independent_acceptance_test_executed": false,
  "pilot50_used_for_training_or_checkpoint_selection": false,
  "source": "BASELINE_SENA_V1_NOT_MISSING_PREVIOUS_CANDIDATE",
  "started_at": "2026-10-19T02:07:36.043289+00:00",
  "trainable_parameters": 14770181,
  "training_records": 648,
  "validation_records": 146,
  "epoch": 3,
  "step": 80,
  "steps_in_epoch": 81,
  "loss": 0.5328494310379028,
  "best_epoch": 2,
  "finished_at": "2026-10-19T02:12:47.255904+00:00",
  "epochs_completed": 3,
  "best_development_selection": 0.6093189964157706,
  "model_exported": true,
  "quality_acceptance_approved": false
}
{
  "accuracy": 0.8561643835616438,
  "macro_f1": 0.8524114362468668,
  "per_class": {
    "FLOOD": {
      "support": 32,
      "precision": 0.8571428571428571,
      "recall": 0.9375,
      "f1": 0.8955223880597015
    },
    "EARTHQUAKE": {
      "support": 24,
      "precision": 0.8461538461538461,
      "recall": 0.916666